# Xây dựng tập dữ liệu OOD AI

### 1. Mục tiêu
Xây dựng quy trình tự động hóa việc làm sạch dữ liệu báo chí (Human-written) và sử dụng các mô hình ngôn ngữ lớn (LLMs) để tạo ra các bài viết tương ứng (AI-generated) nhằm phục vụ huấn luyện mô hình phân biệt văn bản AI.

### 2. Công cụ & Phương pháp
- **Xử lý dữ liệu:** Sử dụng `pandas`, `re` để làm sạch văn bản từ Thanh Niên và VnExpress.
- **LLMs Orchestration:** Sử dụng cơ chế **Fallback Strategy** đa nền tảng (Gemini, Groq, DeepSeek) để tối ưu hóa tốc độ và xử lý lỗi Rate Limit/Quota tự động.
- **Prompt Engineering:** Sử dụng hai cấp độ prompt (`SIMPLE` và `STRICT`) để đa dạng hóa văn phong AI.

### 3. Đầu ra
- `human_news_dataset.csv`: Tập dữ liệu tin tức gốc đã được làm sạch.
- `ai_news_dataset.csv`: Tập dữ liệu song song bao gồm nội dung gốc và nội dung do AI viết lại/mở rộng.

In [ ]:
!pip install python-dotenv huggingface_hub datasets underthesea langdetect google-genai

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.*")

In [ ]:
import gc
import glob
import os
import random
import re
import time
import zlib
from abc import ABC, abstractmethod
from collections import Counter

import numpy as np
import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from google import genai
from huggingface_hub import InferenceClient, login
from langdetect import detect
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
root_path = "/content/drive/MyDrive/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/"
%cd {root_path}
!pwd

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode
/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode


In [ ]:
env_path = find_dotenv()
print(env_path)

if env_path:
    load_dotenv(env_path)
else:
    print("Không tìm thấy file .env!")

/content/drive/.shortcut-targets-by-id/17JBaRqvfDFY0j8cN0vwu8kzqwtDISdDR/BTL_AI_13_IT2302/AIWritingIndicator-13/SourceCode/.env


In [ ]:
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login()
else:
    print("Không tồn tại env HF_TOKEN!")

In [ ]:
class DataLoader:
    @staticmethod
    def load_from_directory(directory_path, file_pattern):
        search_path = os.path.join(directory_path, file_pattern)
        file_list = glob.glob(search_path)

        if not file_list:
            return pd.DataFrame(columns=['source', 'content', 'url'])

        df_list = []
        for file in file_list:
            try:
                df = pd.read_csv(file, usecols=['source', 'content', 'url'])
                df_list.append(df)
            except Exception as e:
                print(f"Lỗi khi đọc file {file}: {e}")

        if not df_list:
            return pd.DataFrame(columns=['source', 'content', 'url'])

        merged_df = pd.concat(df_list, ignore_index=True)
        merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)
        return merged_df

class TextCleaner:
    @staticmethod
    def clean_text(text):
        if pd.isna(text) or not isinstance(text, str) or not text.strip():
            return ""

        text = text.replace('\xa0', ' ').replace('\u200b', '')
        text = re.sub(r'(?:>>\s*)?(?:Bài viết|Ý kiến|Quan điểm).*?không nhất thiết trùng với quan điểm.*$', '', text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r'(?:^|\.\s+)(?:>>\s*)?(?:xem thêm|đọc thêm|tin liên quan|mời xem tiếp|bài liên quan)\b.*?(?:\.|$)', '.', text, flags=re.IGNORECASE)
        text = re.sub(r'Độc giả tìm thêm thông tin chi tiết tại.*$', '', text, flags=re.IGNORECASE | re.DOTALL)
        text = re.sub(r'\(Theo\s.*?\)$|Nguồn:.*$|Ảnh:.*$|Bài, ảnh:.*$|Đồ họa:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
        text = re.sub(r'^[A-Z\s]{2,15}\s?-\s?', '', text)
        text = re.sub(r'\[(VIDEO|CLIP|ẢNH)\][^A-ZÀ-Ỹ]*', '', text)
        text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,7}\b', '', text)
        text = re.sub(r'(\+84|0)\s?\d{2,4}[-\s]?\d{3,4}[-\s]?\d{3,4}', '', text)
        text = re.sub(r'https?://\S+|www\.\S+|[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '', text)
        text = re.sub(r'>{2,}', '', text)
        text = re.sub(r',\s*\.', '.', text)
        text = re.sub(r'\.{4,}', '...', text)
        text = re.sub(r'\|\s*', '', text)
        text = re.sub(r'([a-zà-ỹ])\.([A-ZÀ-Ỹ])', r'\1. \2', text)
        text = re.sub(r'([a-zà-ỹ]),([A-ZÀ-Ỹa-zà-ỹ])', r'\1, \2', text)
        text = re.sub(r'\s+([.,;:?!])', r'\1', text)
        text = re.sub(r'\s+\)', ')', text)
        text = re.sub(r'\(\s+', '(', text)
        text = re.sub(r'\.\s+\.\s+\.', '...', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        text = re.sub(r'(?:\.\s+)(?:[A-Z][a-zA-Z\s&,]+ (?:Resort|Hotel|Spa|Group)(?:,\s*[A-Za-z\s]+)?)$', '.', text)
        return text.strip()

    @classmethod
    def apply_all(cls, df):
        df['content'] = df['content'].apply(cls.clean_text)
        return df

class QualityFilter:
    def filter_by_length(self, df, min_words=200, max_words=800):
        word_counts = df['content'].apply(lambda x: len(str(x).split()))
        return df[(word_counts >= min_words) & (word_counts <= max_words)].copy()

class DatasetPipeline:
    def __init__(self):
        self.quality_filter = QualityFilter()

    def process(self, directory_path, file_pattern):
        df = DataLoader.load_from_directory(directory_path, file_pattern)
        if df.empty:
            return df

        df = df.dropna(subset=['source', 'content', 'url'])
        df = df.drop_duplicates(subset=['content'])
        df = TextCleaner.apply_all(df)
        df = df.replace('', pd.NA).dropna(subset=['content'])
        df = self.quality_filter.filter_by_length(df, 200, 600)
        df = df.drop_duplicates(subset=['content'])

        df = df.reset_index(drop=True)
        return df

In [ ]:
pipeline = DatasetPipeline()

thanhnien_dir = 'Data/data_crawl/news/ThanhNien/'
vnexpress_dir = 'Data/data_crawl/news/VnExpress/'
df_thanhnien = pipeline.process(thanhnien_dir, 'vietnamese_thanhnien_*.csv')
df_vnexpress = pipeline.process(vnexpress_dir, 'vietnamese_vnexpress_*.csv')

if not df_thanhnien.empty and not df_vnexpress.empty:
    final_dataset = pd.concat([df_thanhnien, df_vnexpress], ignore_index=True)
    final_dataset = final_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nHoàn tất! Tổng số bài viết sạch thu được: {len(final_dataset)}")
    display(final_dataset.head())

    final_dataset.to_csv('Data/human_news_dataset.csv', index=False)
    print("Đã lưu file thành công!")
else:
    print("Có lỗi xảy ra: Dữ liệu bị rỗng. Vui lòng kiểm tra lại đường dẫn thư mục.")

Đang xử lý dữ liệu Thanh Niên...
Đang xử lý dữ liệu VnExpress...

Hoàn tất! Tổng số bài viết sạch thu được: 3911


,source,content,url
0,VnExpress,Đội điều tra đặc biệt thuộc Cục Điều tra Quốc ...,https://vnexpress.net/canh-sat-trieu-tap-thu-t...
1,VnExpress,Phát ngôn viên Bộ Kinh tế Đức hôm nay thông bá...,https://vnexpress.net/duc-dieu-tra-vu-sut-ap-d...
2,VnExpress,"Ngày 21/5, bà Vân bị Công an TP HCM ra quyết đ...",https://vnexpress.net/mat-tich-sau-khi-lam-gia...
3,VnExpress,Cổ phiếu của Tập đoàn Vingroup (VIC) sáng nay ...,https://vnexpress.net/vingroup-lap-ky-luc-von-...
4,VnExpress,Trả lời: Sụn chêm có vai trò quan trọng trong ...,https://vnexpress.net/co-nen-cat-bo-sun-chem-5...


Đã lưu file thành công!


### 4. Cơ chế tạo văn bản AI và vòng API Key

**1. Cơ chế tạo văn bản:**
- Sử dụng mô hình AI để **tái cấu trúc (Rewrite)** và **mở rộng (Expand)** từ văn bản gốc của con người.
- Mục tiêu là tạo ra các cặp dữ liệu song hành (Parallel Data) có cùng nội dung cốt lõi nhưng khác biệt hoàn toàn về đặc điểm ngôn ngữ (AI vs Human).

**2. Chiến lược vận hành API Key miễn phí:**
- **Fallback & Rotation:** Do các key miễn phí (Gemini, Groq, DeepSeek) có giới hạn (Rate Limit) rất thấp, hệ thống sử dụng danh sách nhiều key xoay vòng liên tục.
- **Auto-Switching:** Khi một key hoặc model bị lỗi quota/limit, hệ thống tự động chuyển sang model dự phòng hoặc tạm dừng (Penalty time) để tránh làm gián đoạn quá trình xử lý dataset lớn.
- **Error Handling:** Tự động phát hiện và loại bỏ các key đã hết hạn hoặc bị lỗi liên tục để đảm bảo tiến độ.

**3. Tại sao phải đầu tư Prompt Engineering?**
- **Đa dạng hóa phong cách:** Sử dụng nhiều cấp độ prompt giúp AI thoát khỏi các cấu trúc câu lặp lại nhàm chán, tạo ra văn bản có độ biến thiên (Burstiness) cao.
- **Kiểm soát định dạng:** Đảm bảo đầu ra sạch (Clean Output), không chứa các từ thừa của AI (như "Dưới đây là bài viết...") để dữ liệu huấn luyện đạt chất lượng tốt nhất.
- **Gia tăng độ khó:** Prompt phức tạp buộc AI sử dụng từ vựng hiếm và cấu trúc câu đa dạng, giúp mô hình phân biệt sau này trở nên nhạy bén hơn.

In [ ]:
SIMPLE_PROMPT = """Vai trò: Bạn là một nhà báo và biên tập viên chuyên nghiệp.
Nhiệm vụ: Tinh chỉnh và mở rộng văn bản gốc.
Yêu cầu cốt lõi:
- Giữ lại 20-40% ý tưởng/từ khóa gốc; viết lại hoặc sáng tạo 60-80% phần còn lại để bài viết tự nhiên và sắc sảo hơn.
- Định dạng: Chỉ viết đúng MỘT ĐOẠN VĂN DUY NHẤT, không xuống dòng, không danh sách.
- Độ dài: Từ 200 đến 600 từ.
- Ngôn ngữ: Sử dụng 100% câu trần thuật, khách quan, không có câu hỏi tu từ.
- Clean Output: Không in đậm, không in nghiêng, không có câu chào hỏi hay tóm tắt của AI.

Văn bản gốc:
{original_text}"""

STRICT_PROMPT = """Bạn là một nhà báo điều tra và biên tập viên tin tức chuyên nghiệp, có văn phong viết tin tức tự nhiên, sắc sảo và khách quan. Nhiệm vụ của bạn là TINH CHỈNH VÀ MỞ RỘNG văn bản gốc. Bắt buộc phải giữ lại khoảng 20% đến 40% các từ khóa quan trọng, xương sống cú pháp hoặc ý tưởng cốt lõi của tác giả con người. Hãy linh hoạt viết lại toàn bộ phần còn lại (60-80%) để khắc phục các lỗi diễn đạt của văn bản gốc, tạo ra một bản tin sắc sảo, tự nhiên và trôi chảy nhất có thể.

TUYỆT ĐỐI TUÂN THỦ CÁC QUY TẮC SAU (Nếu vi phạm, hệ thống sẽ lỗi):
1. Định dạng Khắc nghiệt:
 Độ dài: Bắt buộc từ 200 đến 600 từ.
 Cấu trúc: Phải là MỘT ĐOẠN VĂN LIỀN MẠCH DUY NHẤT. Tuyệt đối không sử dụng ký tự xuống dòng (\n), không chia đoạn, không dùng gạch đầu dòng hay danh sách.

2. Nhịp điệu và Cấu trúc câu (Burstiness & Clause):
 Tạo ra sự bất ngờ trong độ dài câu. Hãy xen kẽ những câu đơn cực kỳ ngắn, cụt ngủn (3-6 chữ) với những câu ghép rất dài, nhiều mệnh đề phức tạp. Không được viết các câu có độ dài đều đều nhau.
 Không sử dụng câu hỏi tu từ ở bất kỳ đâu trong bài (đặc biệt là mở bài và kết bài). 100% sử dụng câu trần thuật.

3. Từ vựng và Độ cụ thể (Hapax Legomena & Specificity):
 Sử dụng vốn từ vựng phong phú, đưa vào các từ ngữ hiếm gặp, từ chuyên ngành hoặc từ Hán Việt phù hợp để tăng tính đa dạng. Tránh lặp lại một từ khóa quá 3 lần.
 Văn bản phải đậm đặc dữ kiện. Hãy bổ sung (hoặc giữ nguyên) các danh từ riêng, tên người, địa danh, mốc thời gian, số liệu tài chính hoặc phần trăm cụ thể. Không được viết chung chung, mơ hồ.

4. Loại bỏ Thói quen của AI (Transition & Sentiment):
 Giọng văn phải lạnh lùng, khách quan, hoàn toàn vắng bóng cảm xúc cá nhân.
 HẠN CHẾ TỐI ĐA các từ nối chuyển ý mượt mà như: "Tuy nhiên", "Mặt khác", "Hơn thế nữa", "Đáng chú ý", "Tóm lại", "Nhìn chung". Thay vào đó, hãy chuyển ý trực tiếp bằng dữ kiện hoặc dùng dấu chấm phẩy (;).

5. Không có dữ liệu rác (Artifacts):
 Chỉ in ra đúng nội dung bài báo. Tuyệt đối không có các câu mở đầu/kết thúc dạng giao tiếp như: "Dưới đây là bài viết của bạn...", "Chắc chắn rồi...", "Tóm lại, bản tin này..." hay bất kỳ định dạng Markdown in đậm/in nghiêng nào.

6. Cấu trúc Cú pháp và Từ loại (POS Diversity):
 Tuyệt đối tránh việc bắt đầu mọi câu bằng Danh từ hoặc Đại từ. Hãy liên tục thay đổi cấu trúc câu: sử dụng đảo ngữ, câu bị động đan xen chủ động, và đưa các trạng ngữ hoặc cụm giới từ chỉ thời gian/nơi chốn lên đầu câu.

7. Mật độ thông tin (GZIP Anti-compression):
 Cấm sử dụng các cấu trúc câu song song, đối xứng (ví dụ: cấm dùng "Không những... mà còn...", "Một mặt... mặt khác..."). Văn bản phải cô đặc, gãy gọn. Xóa bỏ mọi cụm từ rườm rà không mang thêm thông tin mới.

Văn bản gốc cần xử lý:
{original_text}"""

In [ ]:
CONFIG = {
    "GLOBAL": {
        "MAX_CONSECUTIVE_ERRORS": 10,
        "DEFAULT_PENALTY_SECONDS": 20.0,
    },
   "GEMINI": {
        "MODELS": [
            "gemini-2.5-pro",
            "gemini-2.5-flash",
            "gemini-2.5-flash-lite",
            "gemini-2.0-flash",
            "gemini-2.0-flash-lite",
            "gemini-3-pro-preview",
            "gemini-3.1-pro-preview",
            "gemini-3-flash-preview",
            "gemini-3.1-flash-lite-preview"
        ],
        "MIN_INTERVAL": {
            "gemini-2.5-pro": 12.0,
            "gemini-2.5-flash": 6.0,
            "gemini-2.5-flash-lite": 4.0,
            "gemini-2.0-flash": 4.0,
            "gemini-2.0-flash-lite": 3.0,
            "gemini-3-pro-preview": 12.0,
            "gemini-3.1-pro-preview": 12.0,
            "gemini-3-flash-preview": 6.0,
            "gemini-3.1-flash-lite-preview": 4.0
        },
        "MAX_RATE_LIMIT_RETRIES": 3
    },
    "GROQ": {
        "DEFAULT_MODEL": "llama-3.1-8b-instant",
        "URL": "https://api.groq.com/openai/v1/chat/completions",
        "MIN_INTERVAL": 16.0,
    },
    "DEEPSEEK": {
        "DEFAULT_MODEL": "deepseek-chat",
        "URL": "https://api.deepseek.com/chat/completions",
        "MIN_INTERVAL": 2.0,
    }
}

In [ ]:
class GeminiStrategy:
    def __init__(self, api_key, strategy_name="Gemini"):
        self.api_key = api_key
        self.client = genai.Client(api_key=api_key)
        self.models = CONFIG["GEMINI"]["MODELS"]
        self.current_model_idx = 0
        self.model_name = self.models[self.current_model_idx]
        self.name = strategy_name
        self.provider = "Gemini"
        self.min_interval = CONFIG["GEMINI"]["MIN_INTERVAL"].get(self.model_name, 6.0)
        self.consecutive_errors = 0
        self.rate_limit_count = 0
        self.active = True

    def update_model(self):
        self.current_model_idx += 1
        if self.current_model_idx < len(self.models):
            self.model_name = self.models[self.current_model_idx]
            self.min_interval = CONFIG["GEMINI"]["MIN_INTERVAL"].get(self.model_name, 6.0)
            return True
        return False

    def generate(self, text, prompt_template):
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt_template.format(original_text=text)
        )
        return response.text.strip()

class GroqStrategy:
    def __init__(self, api_key, strategy_name="Groq", model_name=None):
        self.api_key = api_key
        self.model_name = model_name or CONFIG["GROQ"]["DEFAULT_MODEL"]
        self.url = CONFIG["GROQ"]["URL"]
        self.name = strategy_name
        self.provider = "Groq"
        self.min_interval = CONFIG["GROQ"]["MIN_INTERVAL"]
        self.consecutive_errors = 0
        self.active = True

    def generate(self, text, prompt_template):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": self.model_name,
            "messages": [
                {"role": "user", "content": prompt_template.format(original_text=text)}
            ]
        }
        response = requests.post(self.url, headers=headers, json=data)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content'].strip()


class DeepSeekStrategy:
    def __init__(self, api_key, strategy_name="DeepSeek", model_name=None):
        self.api_key = api_key
        self.model_name = model_name or CONFIG["DEEPSEEK"]["DEFAULT_MODEL"]
        self.url = CONFIG["DEEPSEEK"]["URL"]
        self.name = strategy_name
        self.provider = "DeepSeek"
        self.min_interval = CONFIG["DEEPSEEK"]["MIN_INTERVAL"]
        self.consecutive_errors = 0
        self.active = True

    def generate(self, text, prompt_template):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": self.model_name,
            "messages": [
                {"role": "user", "content": prompt_template.format(original_text=text)}
            ]
        }
        response = requests.post(self.url, headers=headers, json=data)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content'].strip()

In [ ]:
class DatasetFallbackGenerator:
    def __init__(self, strategies):
        self.strategies = strategies
        self.next_available_time = {s.name: 0.0 for s in strategies}

    def process_row(self, text, prompt_template):
        while True:
            active_strategies = [s for s in self.strategies if s.active]

            if not active_strategies:
                return "Failed", "All_Models_Failed"

            now = time.time()
            available_strategies = [s for s in active_strategies if now >= self.next_available_time[s.name]]

            if not available_strategies:
                wait_times = [self.next_available_time[s.name] - now for s in active_strategies]
                sleep_time = max(0, min(wait_times))
                if sleep_time > 0:
                    time.sleep(sleep_time)
                continue

            strategy = available_strategies[0]

            self.strategies.remove(strategy)
            self.strategies.append(strategy)

            try:
                result = strategy.generate(text, prompt_template)
                self.next_available_time[strategy.name] = time.time() + strategy.min_interval
                strategy.consecutive_errors = 0

                if isinstance(strategy, GeminiStrategy):
                    strategy.rate_limit_count = 0

                if result:
                    return result, f"{strategy.provider}_{strategy.model_name}"

            except Exception as e:
                strategy.consecutive_errors += 1
                error_str = str(e).lower()

                if strategy.consecutive_errors >= CONFIG["GLOBAL"]["MAX_CONSECUTIVE_ERRORS"]:
                    strategy.active = False
                    tqdm.write(f"[{strategy.name}] Drop do {CONFIG['GLOBAL']['MAX_CONSECUTIVE_ERRORS']} lỗi liên tục.")
                    continue

                if isinstance(strategy, DeepSeekStrategy) and ('quota' in error_str or '402' in error_str):
                    strategy.active = False
                    tqdm.write(f"[{strategy.name}] Hết Quota. Drop key lập tức.")
                    continue

                if isinstance(strategy, GeminiStrategy):
                    if '429' in error_str or 'rate limit' in error_str or 'quota' in error_str:
                        strategy.rate_limit_count += 1
                        if strategy.rate_limit_count >= CONFIG["GEMINI"]["MAX_RATE_LIMIT_RETRIES"] or "you've reached your rate limit" in error_str:
                            has_next = strategy.update_model()
                            strategy.rate_limit_count = 0
                            if not has_next:
                                strategy.active = False
                                tqdm.write(f"[{strategy.name}] Hết toàn bộ model Gemini. Drop key.")
                            else:
                                tqdm.write(f"[{strategy.name}] Chuyển sang model {strategy.model_name} (Delay: {strategy.min_interval}s).")
                            continue

                if '429' in error_str or 'too many requests' in error_str or 'tokens per minute' in error_str or 'rate limit' in error_str:
                    match = re.search(r'try again in ([0-9.]+)s', error_str)
                    if match:
                        penalty = float(match.group(1)) + 1.0
                    else:
                        penalty = CONFIG["GLOBAL"]["DEFAULT_PENALTY_SECONDS"]
                    self.next_available_time[strategy.name] = time.time() + penalty
                else:
                    self.next_available_time[strategy.name] = time.time() + strategy.min_interval

                continue

    def process_dataset(self, input_csv, output_csv, sample_size=None, auto_save_step=100, strict_prompt="", simple_prompt=""):
        if not os.path.exists(input_csv):
            print(f"Lỗi: Không tìm thấy file input gốc tại {input_csv}")
            return

        df_final = pd.read_csv(input_csv)

        if 'ai_generated_content' not in df_final.columns:
            df_final['ai_generated_content'] = ""
        if 'generated_by' not in df_final.columns:
            df_final['generated_by'] = ""

        if os.path.exists(output_csv):
            try:
                df_output = pd.read_csv(output_csv)
                df_final.update(df_output)
                print(f"Đã nạp dữ liệu cũ. Đang tìm các dòng còn thiếu trong tổng số {len(df_final)} dòng...")
            except Exception as e:
                print(f"Không thể nạp file cũ: {e}")

        df_final['ai_generated_content'] = df_final['ai_generated_content'].fillna("")
        df_final['generated_by'] = df_final['generated_by'].fillna("")

        if sample_size:
            df_final = df_final.head(sample_size).copy()

        mask_to_process = (df_final['ai_generated_content'] == "") | (df_final['generated_by'] == "All_Models_Failed")
        df_to_process = df_final[mask_to_process]

        total_rows = len(df_final)
        items_to_process = len(df_to_process)
        items_done = total_rows - items_to_process

        if items_to_process == 0:
            print(f"Tuyệt vời! Đã kiểm tra toàn bộ {total_rows} dòng và tất cả đều đã hoàn thành.")
            return

        print(f"Trạng thái: Đã xong {items_done}/{total_rows} dòng. Còn {items_to_process} dòng cần xử lý.")

        strict_threshold = int(total_rows * 0.65)
        save_counter = 0

        with tqdm(total=items_to_process, desc="AI Generating", bar_format="{l_bar}{bar} | {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
            for index, row in df_to_process.iterrows():
                text = row['content']
                current_prompt = strict_prompt if index > strict_threshold else simple_prompt

                result, model_name = self.process_row(text, current_prompt)

                df_final.at[index, 'ai_generated_content'] = result
                df_final.at[index, 'generated_by'] = model_name

                save_counter += 1
                if save_counter % auto_save_step == 0:
                    temp_file = output_csv + ".tmp"
                    df_final.to_csv(temp_file, index=False, encoding='utf-8')
                    os.replace(temp_file, output_csv)

                pbar.update(1)

        df_final.to_csv(output_csv, index=False, encoding='utf-8')
        print(f"Hoàn thành! File đã được cập nhật đủ dữ liệu tại: {output_csv}")


def load_strategies_from_env():
    strategies = []

    groq_keys_str = os.getenv("GROQ_KEY", "")
    groq_keys = [k.strip() for k in groq_keys_str.split(",")] if groq_keys_str else []
    for idx, key in enumerate(groq_keys, 1):
        if key:
            strategies.append(GroqStrategy(api_key=key, strategy_name=f"Groq_Key{idx}"))

    gemini_keys_str = os.getenv("GOOGLE_GEMINI_KEY", "")
    gemini_keys = [k.strip() for k in gemini_keys_str.split(",")] if gemini_keys_str else []
    for idx, key in enumerate(gemini_keys, 1):
        if key:
            strategies.append(GeminiStrategy(api_key=key, strategy_name=f"Gemini_Key{idx}"))

    deepseek_keys_str = os.getenv("DEEPSEEK_KEY", "")
    deepseek_keys = [k.strip() for k in deepseek_keys_str.split(",")] if deepseek_keys_str else []
    for idx, key in enumerate(deepseek_keys, 1):
        if key:
            strategies.append(DeepSeekStrategy(api_key=key, strategy_name=f"DeepSeek_Key{idx}"))

    return strategies

In [ ]:
active_strategies = load_strategies_from_env()

required_env_vars = [
    "GROQ_KEY",
    "GOOGLE_GEMINI_KEY",
    "DEEPSEEK_KEY",
]

missing_vars = []
for var in required_env_vars:
    value = os.getenv(var)
    if not value or not value.strip():
        missing_vars.append(var)

if missing_vars:
    raise ValueError(
        f"Lỗi cấu hình: Thiếu các biến môi trường sau trong file .env: {', '.join(missing_vars)}. "
        f"Vui lòng kiểm tra lại file cấu hình của bạn."
    )

if not active_strategies:
    print("Không tìm thấy API Key nào. Hãy kiểm tra lại file .env")
else:
    import time

    groq_count = sum(1 for s in active_strategies if s.provider == "Groq")
    gemini_count = sum(1 for s in active_strategies if s.provider == "Gemini")
    deepseek_count = sum(1 for s in active_strategies if s.provider == "DeepSeek")

    print("\n" + "="*35)
    print("TỔNG HỢP API KEYS ĐANG HOẠT ĐỘNG")
    print("="*35)
    print(f"Groq     : {groq_count} keys")
    print(f"Gemini   : {gemini_count} keys")
    print(f"DeepSeek : {deepseek_count} keys")
    print(f"Tổng cộng: {len(active_strategies)} keys")
    print("="*35)
    print("⏳ Hệ thống sẽ bắt đầu xử lý sau 10 giây...\n")

    time.sleep(10)

    generator = DatasetFallbackGenerator(strategies=active_strategies)
    generator.process_dataset(
        input_csv="Data/human_news_dataset.csv",
        output_csv="Data/ai_news_dataset.csv",
        auto_save_step=10,
        strict_prompt=STRICT_PROMPT,
        simple_prompt=SIMPLE_PROMPT
    )


TỔNG HỢP API KEYS ĐANG HOẠT ĐỘNG
Groq     : 3 keys
Gemini   : 16 keys
DeepSeek : 4 keys
Tổng cộng: 23 keys
⏳ Hệ thống sẽ bắt đầu xử lý sau 10 giây...

🔄 Đã nạp dữ liệu cũ. Đang tìm các dòng còn thiếu trong tổng số 3911 dòng...
📊 Trạng thái: Đã xong 1310/3911 dòng. Còn 2601 dòng cần xử lý.


AI Generating:   0%|           | 3/2601 [00:07<49:04,  1.13s/it]

❌ [DeepSeek_Key1] Hết Quota. Drop key lập tức.


AI Generating:   0%|           | 3/2601 [00:08<49:04,  1.13s/it]

❌ [DeepSeek_Key2] Hết Quota. Drop key lập tức.


...

🔄 [Gemini_Key15] Chuyển sang model gemini-3-pro-preview (Delay: 12.0s).
❌ [Gemini_Key15] Drop do 10 lỗi liên tục.


AI Generating:  35%|███▍       | 903/2601 [2:12:31<3:25:26,  7.26s/it]

❌ [Groq_Key1] Drop do 10 lỗi liên tục.


AI Generating:  40%|████       | 1043/2601 [2:32:29<3:47:46,  8.77s/it]


KeyboardInterrupt: 